# wrangle / portfolio / withdrawal

**Intent**
- Estimates `portfolio_withdrawal_rate_pct` from `portfolio_withdrawal_age_years`, and must:
  - Give the sustainable withdrawal rate, the percent of wealth withdrawn each year in real terms.
  - Take it from a risk-free return to the planning horizon.
  - Be the ceiling any risky mix has to beat.
  - Hold the rmse on `validation` and `backtest` to 0.01.

**Data**
- Personal data is git-ignored in `data/`, with synthetic placeholders committed.
- `training` is ten ages with the rate, `validation` is five further, and `backtest` is ten across the range.
- `backtest` is held back for acceptance.
- `golden` is ages 30 to 120, then a request with the input missing.

**Status**
- The real return is recovered from synthetic points, a placeholder of 2 percent to age 100.

**Future**
- Overall: the work still to do before the first release:
  - Promote the target from local to served, then add the Register stage, once the limits are fixed against real data.
  - Add `query_postgres` to `collect_run` and load the real series in place of the synthetic ones.
  - Use the annuity rate `r / (1 - (1 + r)^-n)`, with `r` from the index linked bond yield.
  - Replace the fixed horizon with the survival curve from ABS life tables, joint with your partner.
  - Add the Age Pension and superannuation preservation as an income floor.
  - Fix the limits against real data, then register and review before any release.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("wrangle/portfolio/withdrawal")


GOLDEN = "portfolio_withdrawal_golden"

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run

HORIZON = 100.0
ws.params = {"horizon": HORIZON}


def rate(age, real=0.02):
    return 100.0 * real / (1.0 - (1.0 + real) ** -max(HORIZON - age, 1.0))


def labelled(ages):
    return {ws.inputs[0]: ages, ws.outputs[0]: [round(rate(age), 6) for age in ages]}


collect_run(
    ws, 1000, label="Ten synthetic ages with the rate", name="portfolio_withdrawal_training", source="builder", build=lambda: labelled([40.0, 45.0, 50.0, 55.0, 60.0, 65.0, 70.0, 75.0, 80.0, 85.0])
)
collect_run(ws, 1000, label="Five further synthetic ages", name="portfolio_withdrawal_validation", source="builder", build=lambda: labelled([42.0, 52.0, 62.0, 72.0, 82.0]))
collect_run(
    ws, 1000, label="Ten synthetic ages across the range", name="portfolio_withdrawal_backtest", source="builder", build=lambda: labelled([41.0, 47.0, 53.0, 59.0, 67.0, 73.0, 79.0, 83.0, 87.0, 90.0])
)

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "portfolio_withdrawal_training"), validation=ws.run(1000, "portfolio_withdrawal_validation"), backtest=ws.run(1000, "portfolio_withdrawal_backtest"))
print("training", ws.training.height, "validation", ws.validation.height, "backtest", ws.backtest.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
from scipy.optimize import curve_fit

HORIZON = ws.params["horizon"]
fitted_real = curve_fit(
    lambda age, real: 100.0 * real / (1.0 - (1.0 + real) ** -(HORIZON - age)), ws.training[ws.inputs[0]].to_numpy(), ws.training[ws.outputs[0]].to_numpy(), p0=[0.03], bounds=(0.001, 0.2)
)[0][0]
ws.params = {"real": round(float(fitted_real), 4), "horizon": HORIZON}
print("fitted", ws.params)
assert ws.params == {"real": 0.02, "horizon": 100.0}, "the fit did not recover the real return"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

REAL, HORIZON = ws.params["real"], ws.params["horizon"]


def withdrawal(age):
    return 100.0 * REAL / (1.0 - (1.0 + REAL) ** -max(HORIZON - age, 1.0))


define_model(ws, model=1000, flavour="pyfunc", source=withdrawal, label="Annuity withdrawal rate, least squares real return", constants={"REAL": ws.params["real"], "HORIZON": ws.params["horizon"]})

## Evaluate ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-evaluate))

In [ ]:
from mlmodels import evaluate_gate, evaluate_metrics, evaluate_model, evaluate_models

LIMITS = dict.fromkeys(ws.outputs, 0.01)


def acceptance_checks():
    checks = []
    for split in ("validation", "backtest"):
        frame = getattr(ws, split)
        predicted = evaluate_model(ws, 1000, split)
        checks.extend(
            {
                "split": split,
                "gate": split == "backtest",
                "output": output,
                "name": "rmse",
                "value": evaluate_metrics(frame[output], predicted[output])["rmse"],
                "limit": LIMITS[output],
                "direction": "max",
            }
            for output in ws.outputs
        )
    return checks


evaluate_gate(ws, 1000, acceptance_checks)
evaluate_models(ws)

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = {ws.inputs[0]: [30.0, 50.0, 65.0, 80.0, 99.5, 120.0, None], "expect": ["ok"] * 6 + ["error"]}
package_model(ws, 1000, golden, target={"kind": "local"})